# Shor's Algorithm from Scratch, Part 2: Modular Adder

In [Part 1](401_shor_scratch_adder.ipynb), we built an ordinary quantum adder. This time we implement **modular** addition $a+b \bmod N$ on top of it — this is the operation actually needed inside Shor's algorithm, because every intermediate value that arises while computing $x^j \bmod N$ must stay within $[0, N)$.

Reference: V. Vedral, A. Barenco, A. Ekert, ["Quantum Networks for Elementary Arithmetic Operations"](https://arxiv.org/abs/quant-ph/9511018) (1995), Section III.B (Figure 4)

## The 5-step idea

Classically, $(a+b) \bmod N$ is easy: add, and if the result is $N$ or more, subtract $N$ by that amount. On a quantum computer, however, we cannot "check and then branch" in the usual way (that would require measurement, which destroys the superposition we care about). Instead, the check must be *recorded in qubits*, and the conditional branch must be expressed *in the shape of the circuit*.

1. **Add**: $b \leftarrow a + b$ (using the adder from Part 1)
2. **Subtract $N$**: $b \leftarrow b - N$ (run the adder backwards)
3. **Detect the sign**: if $b$ went negative in step 2, the extra top qubit of the $b$ register becomes $1$ (it acts as a borrow/overflow flag) — copy this bit into a flag qubit $t$
4. **Conditionally add $N$ back**: if $t=1$ (i.e. $a+b < N$ and we subtracted $N$ too much), add $N$ back — but *only* to the branch where $t=1$ within the superposition
5. **Clean up**: return $t$ to $\ket 0$ and undo the sign-detection step. This makes the whole circuit a clean, reusable operation that leaves no trace on anything except $b$

The precondition is $0 \le a, b < N$. Steps 1–2 just use the adder/subtractor as they are. **The interesting part is step 4**, where the paper uses a slightly surprising trick.

In [ ]:
!pip install git+https://github.com/blueqat/blueqatSDK

## Reusing the parts from Part 1

In addition to the same `carry`/`carry_inv`/`sum_gate` as last time, we prepare the assembled full adder `add_circuit` and its inverse `sub_circuit`. `sub_circuit` is obtained simply by running the same gate sequence in reverse order (every gate used here is either `CX` or `CCX`, both of which are self-inverse, so reversing the order is enough — there is no need to invert the individual gates). The paper says the same thing (*an adder whose network is run backwards*).

In [2]:
from blueqat import Circuit

def carry(c, a, b, c_next):
    circ = Circuit()
    circ.ccx[a, b, c_next]
    circ.cx[a, b]
    circ.ccx[c, b, c_next]
    return circ

def carry_inv(c, a, b, c_next):
    circ = Circuit()
    circ.ccx[c, b, c_next]
    circ.cx[a, b]
    circ.ccx[a, b, c_next]
    return circ

def sum_gate(c, a, b):
    circ = Circuit()
    circ.cx[a, b]
    circ.cx[c, b]
    return circ

def add_circuit(c, a, b, n):
    """b := a + b, using carry register c. b must have n+1 qubits."""
    circ = Circuit()
    for i in range(n - 1):
        circ += carry(c[i], a[i], b[i], c[i + 1])
    circ += carry(c[n - 1], a[n - 1], b[n - 1], b[n])
    circ.cx[a[n - 1], b[n - 1]]
    circ += sum_gate(c[n - 1], a[n - 1], b[n - 1])
    for i in range(n - 2, -1, -1):
        circ += carry_inv(c[i], a[i], b[i], c[i + 1])
        circ += sum_gate(c[i], a[i], b[i])
    return circ

def sub_circuit(c, a, b, n):
    """b := b - a: the adder's gate list, reversed."""
    add_circ = add_circuit(c, a, b, n)
    reversed_circ = Circuit()
    reversed_circ.ops = list(reversed(add_circ.ops))
    return reversed_circ

## Step 4: make the addend conditional, not the adder

How do we build "perform $b \leftarrow b + N$ only when $t=1$"?

The straightforward idea is to **control every gate** in `add_circuit(c, N, b, n)` on $t$. Controlling a `CX` with one more qubit gives a `CCX`, which is easy, but controlling a `CCX` further requires a **3-controlled X**, for which this SDK has no direct gate. It can be emulated with ancillas and a chain of Toffolis, but the circuit becomes considerably heavier.

The paper does something completely different.

> Conditionally on the value of this last qubit $\ket t$, $N$ is added back to the second register, leaving it with the value $a + b \bmod N$. This is done by **either leaving the first register with the value $N$** (in case of overflow), **or resetting it to 0** (if there is no overflow) **and then using a plain adder**.

In other words, **we do not touch the adder at all**. Instead, we make the $N$ register that holds the addend conditional. If $t=1$ it stays $N$; if $t=0$ it is reset to $0$, and then we pass it through the **uncontrolled adder** as is.

Why does this suffice? **Because adding $0$ does nothing.** The adder itself can run unconditionally, and if the input is $0$ it simply becomes the identity.

With this construction, **no 3-controlled gate ever appears** anywhere in the circuit. No ancilla qubits are needed either.

$N$ is a classical constant known at circuit-construction time, so which bits are $1$ is already known. We only need to apply $0$-controlled CNOTs — `X` on either side of $t$ — to the bits where $N$ is set.

In [3]:
def set_to_zero_if(t, N, val_N, n):
    """Reset the N register to 0 when t == 0 (leave it alone when t == 1).
    Self-inverse: applying it twice restores N."""
    circ = Circuit()
    circ.x[t]
    for j in range(n):
        if (val_N >> j) & 1:
            circ.cx[t, N[j]]
    circ.x[t]
    return circ

Both `X` and `CX` are self-inverse, and moreover they have different targets, so **running this circuit once more restores $N$**. This is the simplest form of uncomputation, which has appeared repeatedly since Part 1: "apply it, use it, and undo it with the same thing."

## Register layout

The full modular adder needs $4n+2$ qubits.

- `c[0..n-1]`: carry register (shared by the addition and subtraction steps)
- `a[0..n-1]`: the first addend
- `b[0..n]`: register holding the second addend and the result (the same $n+1$ width as in Part 1; in step 3 its top bit doubles as the overflow/borrow flag)
- `N[0..n-1]`: the modulus (loaded as a constant)
- `t`: a single flag qubit indicating "whether $N$ needed to be added back"

No ancilla qubits for multi-controlled gates are required.

In [4]:
def modulo_add_circuit(c, a, b, N, t, n, val_N):
    """b := (a + b) mod N on the given registers -- the five steps, paper Fig.4.
    Carries no input encoding, so it can be reused as a black box."""
    circ = Circuit()

    # Steps 1-2: b <- a + b, then b <- b - N
    circ += add_circuit(c, a, b, n)
    circ += sub_circuit(c, N, b, n)

    # Step 3: copy the borrow/overflow bit into the flag t
    circ.cx[b[n], t]

    # Step 4 (paper, Fig.4): blank the N register when t == 0, then a PLAIN adder,
    # then restore N with the very same gates.
    gate = set_to_zero_if(t, N, val_N, n)
    circ += gate
    circ += add_circuit(c, N, b, n)
    circ += gate

    # Step 5: clean up -- undo the borrow/overflow detection and reset t to |0>
    circ += sub_circuit(c, a, b, n)
    circ.x[b[n]]
    circ.cx[b[n], t]
    circ.x[b[n]]
    circ += add_circuit(c, a, b, n)

    return circ


def build_modulo_adder(input_a, input_b, input_N, n):
    """Encode the inputs, then run the modular adder.
    Returns (circuit, b_register_qubit_indices)."""
    circ = Circuit(4 * n + 2)

    c = list(range(n))
    a = list(range(n, 2 * n))
    b = list(range(2 * n, 3 * n + 1))
    N = list(range(3 * n + 1, 4 * n + 1))
    t = 4 * n + 1

    for i in range(n):
        if (input_a >> i) & 1:
            circ.x[a[i]]
        if (input_b >> i) & 1:
            circ.x[b[i]]
        if (input_N >> i) & 1:
            circ.x[N[i]]

    circ += modulo_add_circuit(c, a, b, N, t, n, input_N)
    return circ, b

## Testing

We use the same bit-order convention as in Part 1: in the result string, qubit `q` is `state[total - 1 - q]` (the rightmost character is qubit 0).

We explicitly specify `statevector` as the simulator backend. blueqat's default runs on tensor-network contraction, so the cost is determined by *how the circuit is connected* rather than by the number of qubits. This is a poor match for this kind of "few qubits but deep" circuit; in Part 3's multiplier it ran out of memory and crashed at barely 20 qubits. With a state vector, memory consumption is straightforwardly determined by $2^{(\text{number of qubits})}$.

In [5]:
def run_modulo_adder(val_a, val_b, val_N, n=None):
    if n is None:
        n = max(val_N, 1).bit_length()
    circuit, b = build_modulo_adder(val_a, val_b, val_N, n)
    state = circuit.m[:].run(shots=1, backend="statevector").most_common(1)[0][0]
    total = circuit.n_qubits
    bits = [state[total - 1 - q] for q in b]  # bits[0] = LSB
    return int("".join(reversed(bits)), 2)

tests = [
    (7, 6, 11, 4),
    (14, 13, 15, 4),
    (3, 5, 9, 4),
    (0, 0, 7, 3),
    (5, 5, 6, 3),
]
for x, y, N, n in tests:
    result = run_modulo_adder(x, y, N, n)
    expected = (x + y) % N
    print(f"({x}+{y}) mod {N} = {result}  (expected {expected}, {'OK' if result == expected else 'MISMATCH'})")

(7+6) mod 11 = 2  (expected 2, OK)


(14+13) mod 15 = 12  (expected 12, OK)


(3+5) mod 9 = 8  (expected 8, OK)


(0+0) mod 7 = 0  (expected 0, OK)


(5+5) mod 6 = 4  (expected 4, OK)


Some cases are chosen to go through both branches: `(7+6) mod 11` does not go negative when $N$ is subtracted ($7+6=13 \ge 11$), so $t=0$, the $N$ register is set to $0$ in step 4, and nothing is added back. On the other hand, `(0+0) mod 7` does go negative when $N$ is subtracted ($0+0 < 7$), so $t=1$ and $N$ is added back in step 4 — in both cases the same circuit works correctly.

## Exhaustive test: do the working qubits return to $\ket 0$?

The modular adder is a component we will reuse dozens of times in later parts. It is not enough for the result $b$ to be correct. **The carry register $c$, the modulus register $N$, and the flag $t$ must all be restored.** If they are not, those bits remain entangled with the result and break the quantum interference in later stages.

For $N = 5, 7, 9$, we try all combinations of $a$ and $b$, checking both the value and the cleanup.

In [6]:
def run_and_check(val_a, val_b, val_N, n=None):
    """Returns (result, everything_else_is_back_to_|0>)."""
    if n is None:
        n = max(val_N, 1).bit_length()
    circuit, b = build_modulo_adder(val_a, val_b, val_N, n)
    c = list(range(n))
    N = list(range(3 * n + 1, 4 * n + 1))
    t = 4 * n + 1
    state = circuit.m[:].run(shots=1, backend="statevector").most_common(1)[0][0]
    total = circuit.n_qubits
    read = lambda qs: int("".join(reversed([state[total - 1 - q] for q in qs])), 2)
    clean = read(N) == val_N and int(state[total - 1 - t]) == 0 and read(c) == 0
    return read(b), clean

bad = 0
for N in (5, 7, 9):
    for a in range(N):
        for b in range(N):
            result, clean = run_and_check(a, b, N)
            if result != (a + b) % N or not clean:
                bad += 1
                print(f"MISMATCH ({a}+{b}) mod {N} -> {result}, clean={clean}")
print(f"155 cases checked, {bad} mismatches")

155 cases checked, 0 mismatches


## Scaling

Both the qubit count and the gate count are **linear** in $n$. And the maximum number of controls is $2$ no matter what — i.e. it never goes beyond Toffoli.

In [7]:
for n in (3, 4, 8, 16):
    c = list(range(n)); a = list(range(n, 2 * n)); b = list(range(2 * n, 3 * n + 1))
    N = list(range(3 * n + 1, 4 * n + 1)); t = 4 * n + 1
    circ = modulo_add_circuit(c, a, b, N, t, n, (1 << n) - 1)
    max_ctrl = max(g.n_qargs - 1 for g in circ.ops)   # X=0, CX=1, CCX=2
    print(f"n={n:2d}: {4 * n + 2:3d} qubits, {len(circ.ops):4d} gates, "
          f"max {max_ctrl} controls")

n= 3:  14 qubits,  124 gates, max 2 controls
n= 4:  18 qubits,  166 gates, max 2 controls
n= 8:  34 qubits,  334 gates, max 2 controls
n=16:  66 qubits,  670 gates, max 2 controls


This matches the paper's statement that "the number of elementary gates in the plain adder, the modular addition and the controlled–modular addition network scales linearly with $n$."

## Next time

Now that we have addition and modular addition, the next step is the [controlled modular multiplier](403_shor_scratch_controlled_multiplier.ipynb) — an operation that multiplies a constant modulo $N$ under the control of a qubit, implemented by repeatedly modular-adding shifted copies of the multiplicand (a reversible pencil-and-paper style multiplication).

There too, exactly the same trick as in step 4 here — **instead of controlling the operation, make the input conditional so the operation becomes trivial** — appears once more. The paper uses this idea in two places, Fig. 4 and Fig. 5.